### Install Required Libraries

In [ ]:
%pip install openpyxl xlsxwriter pandas numpy

### Import Libraries

In [1]:
import pandas as pd
import numpy as np
import os
import glob
from pathlib import Path
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# Configuration paths
BHAVCOPY_DIR = r"C:\Users\bkaly\Documents\Trading\bhavcopy"
FO_DATA_DIR = r"C:\Users\bkaly\Documents\Trading\fo"
OUTPUT_DIR = r"C:\Users\bkaly\Documents\Trading\analytics_output"

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

### Core Functions for Data Loading and Processing

In [2]:
def load_latest_bhavcopy(bhavcopy_dir: str) -> pd.DataFrame:
    """
    Load the latest bhavcopy file from the folder.
    
    Args:
        bhavcopy_dir (str): Path to bhavcopy folder
    
    Returns:
        pd.DataFrame: Combined bhavcopy data
    """
    try:
        # Find all bhavcopy CSV files
        bhavcopy_files = sorted(glob.glob(os.path.join(bhavcopy_dir, "*_bhavcopy.csv")))
        
        if not bhavcopy_files:
            print(f"No bhavcopy files found in {bhavcopy_dir}")
            return pd.DataFrame()
        
        # Load the latest file (last one alphabetically)
        latest_file = bhavcopy_files[-1]
        print(f"Loading: {os.path.basename(latest_file)}")
        
        df = pd.read_csv(latest_file)
        print(f"Loaded {len(df)} records")
        return df
    
    except Exception as e:
        print(f"Error loading bhavcopy: {e}")
        return pd.DataFrame()


def load_previous_bhavcopy(bhavcopy_dir: str, days_back: int = 1) -> pd.DataFrame:
    """
    Load previous day's bhavcopy file.
    
    Args:
        bhavcopy_dir (str): Path to bhavcopy folder
        days_back (int): Days to go back (default 1 for previous day)
    
    Returns:
        pd.DataFrame: Previous day's bhavcopy data
    """
    try:
        bhavcopy_files = sorted(glob.glob(os.path.join(bhavcopy_dir, "*_bhavcopy.csv")))
        
        if len(bhavcopy_files) < days_back + 1:
            print(f"Not enough files for {days_back} days back")
            return pd.DataFrame()
        
        # Load file from days_back
        prev_file = bhavcopy_files[-(days_back + 1)]
        print(f"Loading previous: {os.path.basename(prev_file)}")
        
        df = pd.read_csv(prev_file)
        return df
    
    except Exception as e:
        print(f"Error loading previous bhavcopy: {e}")
        return pd.DataFrame()

### Analytics Calculation Functions

In [3]:
def calculate_options_coi(current_df: pd.DataFrame, previous_df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate Options Change in Open Interest (COI) per symbol and expiry.
    
    Args:
        current_df (pd.DataFrame): Current day bhavcopy data
        previous_df (pd.DataFrame): Previous day bhavcopy data
    
    Returns:
        pd.DataFrame: Options COI aggregated by symbol and expiry
    """
    # Filter for options (FinInstrmTp == 'STO')
    current_opts = current_df[current_df['FinInstrmTp'] == 'STO'].copy()
    previous_opts = previous_df[previous_df['FinInstrmTp'] == 'STO'].copy() if not previous_df.empty else pd.DataFrame()
    
    # Group by symbol and expiry
    current_coi = current_opts.groupby(['TckrSymb', 'XpryDt'])['ChngInOpnIntrst'].sum().reset_index()
    current_coi.columns = ['Symbol', 'Expiry', 'Current_COI']
    
    if not previous_opts.empty:
        previous_coi = previous_opts.groupby(['TckrSymb', 'XpryDt'])['ChngInOpnIntrst'].sum().reset_index()
        previous_coi.columns = ['Symbol', 'Expiry', 'Previous_COI']
        
        # Merge current and previous
        result = current_coi.merge(previous_coi, on=['Symbol', 'Expiry'], how='left')
        result['Previous_COI'].fillna(0, inplace=True)
    else:
        result = current_coi.copy()
        result['Previous_COI'] = 0
    
    # Calculate metrics
    result['COI_Change'] = result['Current_COI'] - result['Previous_COI']
    result['COI_Change_Pct'] = result.apply(
        lambda row: (row['COI_Change'] / row['Previous_COI'] * 100) if row['Previous_COI'] != 0 else np.nan,
        axis=1
    )
    
    return result.sort_values('COI_Change', ascending=False)


def calculate_futures_coi(current_df: pd.DataFrame, previous_df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate Futures Change in Open Interest per symbol and expiry.
    
    Args:
        current_df (pd.DataFrame): Current day bhavcopy data
        previous_df (pd.DataFrame): Previous day bhavcopy data
    
    Returns:
        pd.DataFrame: Futures COI aggregated by symbol and expiry
    """
    # Filter for futures (FinInstrmTp == 'FUT')
    current_fut = current_df[current_df['FinInstrmTp'] == 'FUT'].copy()
    previous_fut = previous_df[previous_df['FinInstrmTp'] == 'FUT'].copy() if not previous_df.empty else pd.DataFrame()
    
    # Group by symbol and expiry
    current_coi = current_fut.groupby(['TckrSymb', 'XpryDt'])['ChngInOpnIntrst'].sum().reset_index()
    current_coi.columns = ['Symbol', 'Expiry', 'Current_COI']
    
    if not previous_fut.empty:
        previous_coi = previous_fut.groupby(['TckrSymb', 'XpryDt'])['ChngInOpnIntrst'].sum().reset_index()
        previous_coi.columns = ['Symbol', 'Expiry', 'Previous_COI']
        
        result = current_coi.merge(previous_coi, on=['Symbol', 'Expiry'], how='left')
        result['Previous_COI'].fillna(0, inplace=True)
    else:
        result = current_coi.copy()
        result['Previous_COI'] = 0
    
    # Calculate metrics
    result['COI_Change'] = result['Current_COI'] - result['Previous_COI']
    result['COI_Change_Pct'] = result.apply(
        lambda row: (row['COI_Change'] / row['Previous_COI'] * 100) if row['Previous_COI'] != 0 else np.nan,
        axis=1
    )
    
    return result.sort_values('COI_Change', ascending=False)


def calculate_put_call_ratio(current_df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate Put-Call Ratio (PCR) based on open interest and volume.
    
    Args:
        current_df (pd.DataFrame): Current day bhavcopy data
    
    Returns:
        pd.DataFrame: PCR metrics aggregated by symbol and expiry
    """
    # Filter for options
    options = current_df[current_df['FinInstrmTp'] == 'STO'].copy()
    
    if options.empty:
        return pd.DataFrame()
    
    # Aggregate by symbol, expiry, and option type (CE/PE)
    oi_data = options.groupby(['TckrSymb', 'XpryDt', 'OptnTp']).agg({
        'OpnIntrst': 'sum',
        'TtlTradgVol': 'sum',
        'ChngInOpnIntrst': 'sum'
    }).reset_index()
    
    # Separate CE and PE
    ce_data = oi_data[oi_data['OptnTp'] == 'CE'].copy()
    pe_data = oi_data[oi_data['OptnTp'] == 'PE'].copy()
    
    # Rename columns for clarity
    ce_data.columns = ['Symbol', 'Expiry', 'OptnTp', 'CE_OI', 'CE_Volume', 'CE_COI']
    pe_data.columns = ['Symbol', 'Expiry', 'OptnTp', 'PE_OI', 'PE_Volume', 'PE_COI']
    
    # Merge CE and PE data
    pcr_data = ce_data[['Symbol', 'Expiry', 'CE_OI', 'CE_Volume', 'CE_COI']].merge(
        pe_data[['Symbol', 'Expiry', 'PE_OI', 'PE_Volume', 'PE_COI']],
        on=['Symbol', 'Expiry'],
        how='outer'
    )
    
    # Fill NaN values with 0
    pcr_data = pcr_data.fillna(0)
    
    # Calculate PCR ratios
    pcr_data['PCR_OI'] = pcr_data.apply(
        lambda row: row['PE_OI'] / row['CE_OI'] if row['CE_OI'] > 0 else 0,
        axis=1
    )
    
    pcr_data['PCR_Volume'] = pcr_data.apply(
        lambda row: row['PE_Volume'] / row['CE_Volume'] if row['CE_Volume'] > 0 else 0,
        axis=1
    )
    
    # PCR COI (Change in Open Interest ratio)
    pcr_data['PCR_COI'] = pcr_data.apply(
        lambda row: row['PE_COI'] / row['CE_COI'] if row['CE_COI'] > 0 else 0,
        axis=1
    )
    
    return pcr_data.sort_values(['Symbol', 'Expiry'])

### Excel Export Functions

In [4]:
def format_excel_sheet(worksheet, data_df: pd.DataFrame):
    """
    Apply formatting to an Excel worksheet.
    
    Args:
        worksheet: OpenPyXL worksheet object
        data_df (pd.DataFrame): DataFrame to format
    """
    # Define styles
    header_fill = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
    header_font = Font(bold=True, color="FFFFFF", size=11)
    border = Border(
        left=Side(style='thin'),
        right=Side(style='thin'),
        top=Side(style='thin'),
        bottom=Side(style='thin')
    )
    
    # Format header row
    for cell in worksheet[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell.border = border
    
    # Format data rows
    for row in worksheet.iter_rows(min_row=2, max_row=worksheet.max_row, min_col=1, max_col=worksheet.max_column):
        for cell in row:
            cell.border = border
            cell.alignment = Alignment(horizontal='center', vertical='center')
            
            # Format numeric columns
            if isinstance(cell.value, (int, float)):
                if 'Pct' in str(cell.column_letter):
                    cell.number_format = '0.00%'
                else:
                    cell.number_format = '#,##0'
    
    # Adjust column widths
    for col_idx, col in enumerate(data_df.columns, 1):
        max_length = max(len(str(data_df[col].max())) if data_df[col].dtype in ['int64', 'float64'] else data_df[col].str.len().max(),
                         len(str(col)))
        worksheet.column_dimensions[get_column_letter(col_idx)].width = min(max_length + 2, 30)


def export_to_excel(output_path: str, options_coi_df: pd.DataFrame, futures_coi_df: pd.DataFrame, pcr_df: pd.DataFrame, timestamp: str):
    """
    Export all analytics to a single Excel file with multiple sheets.
    
    Args:
        output_path (str): Path to save the Excel file
        options_coi_df (pd.DataFrame): Options COI data
        futures_coi_df (pd.DataFrame): Futures COI data
        pcr_df (pd.DataFrame): Put-Call Ratio data
        timestamp (str): Timestamp for the report
    """
    try:
        with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
            # Write Options COI
            options_coi_df.to_excel(writer, sheet_name='Options_COI', index=False)
            
            # Write Futures COI
            futures_coi_df.to_excel(writer, sheet_name='Futures_COI', index=False)
            
            # Write Put-Call Ratio
            pcr_df.to_excel(writer, sheet_name='PCR', index=False)
            
            # Add Summary sheet
            summary_data = {
                'Metric': [
                    'Report Generated',
                    'Total Options Records (COI)',
                    'Total Futures Records (COI)',
                    'Total Symbols (PCR)',
                    'Data Source'
                ],
                'Value': [
                    timestamp,
                    len(options_coi_df),
                    len(futures_coi_df),
                    len(pcr_df),
                    'NSE Bhavcopy Data'
                ]
            }
            summary_df = pd.DataFrame(summary_data)
            summary_df.to_excel(writer, sheet_name='Summary', index=False)
        
        # Apply formatting to all sheets
        from openpyxl import load_workbook
        wb = load_workbook(output_path)
        for sheet_name in wb.sheetnames:
            ws = wb[sheet_name]
            if sheet_name == 'Options_COI':
                format_excel_sheet(ws, options_coi_df)
            elif sheet_name == 'Futures_COI':
                format_excel_sheet(ws, futures_coi_df)
            elif sheet_name == 'PCR':
                format_excel_sheet(ws, pcr_df)
            elif sheet_name == 'Summary':
                format_excel_sheet(ws, summary_df)
        
        wb.save(output_path)
        print(f"\nExcel file saved successfully: {output_path}")
    
    except Exception as e:
        print(f"Error exporting to Excel: {e}")

### Main Execution Function

In [5]:
def run_daily_analytics():
    """
    Main function to run all analytics computations.
    This function can be executed daily to generate updated reports.
    """
    print("="*80)
    print("Starting Daily Options Analytics")
    print("="*80)
    
    # Load data
    print("\n[Step 1] Loading data...")
    current_df = load_latest_bhavcopy(BHAVCOPY_DIR)
    previous_df = load_previous_bhavcopy(BHAVCOPY_DIR, days_back=1)
    
    if current_df.empty:
        print("ERROR: No current data available. Exiting.")
        return
    
    # Calculate analytics
    print("\n[Step 2] Computing Options Change in Open Interest...")
    options_coi = calculate_options_coi(current_df, previous_df)
    print(f"Computed {len(options_coi)} option symbols")
    
    print("\n[Step 3] Computing Futures Change in Open Interest...")
    futures_coi = calculate_futures_coi(current_df, previous_df)
    print(f"Computed {len(futures_coi)} future symbols")
    
    print("\n[Step 4] Computing Put-Call Ratios...")
    pcr = calculate_put_call_ratio(current_df)
    print(f"Computed {len(pcr)} symbol-expiry combinations")
    
    # Generate timestamp for filename
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_filename = f"OptionsAnalytics_{timestamp}.xlsx"
    output_path = os.path.join(OUTPUT_DIR, output_filename)
    
    # Export results
    print("\n[Step 5] Exporting to Excel...")
    export_to_excel(output_path, options_coi, futures_coi, pcr, datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
    
    # Display summary
    print("\n" + "="*80)
    print("SUMMARY - Top 5 Options COI Changes:")
    print("="*80)
    print(options_coi.head(5)[['Symbol', 'Expiry', 'Current_COI', 'COI_Change', 'COI_Change_Pct']].to_string(index=False))
    
    print("\n" + "="*80)
    print("SUMMARY - Top 5 Futures COI Changes:")
    print("="*80)
    print(futures_coi.head(5)[['Symbol', 'Expiry', 'Current_COI', 'COI_Change', 'COI_Change_Pct']].to_string(index=False))
    
    print("\n" + "="*80)
    print("SUMMARY - Top 5 Put-Call Ratios (PCR OI):")
    print("="*80)
    top_pcr = pcr.sort_values('PCR_OI', ascending=False).head(5)
    print(top_pcr[['Symbol', 'Expiry', 'CE_OI', 'PE_OI', 'PCR_OI']].to_string(index=False))
    
    print("\n" + "="*80)
    print("Daily Analytics Complete!")
    print(f"Output saved to: {output_path}")
    print("="*80)
    
    return options_coi, futures_coi, pcr

### Execute Daily Analytics

In [6]:
# Run the daily analytics
options_coi_results, futures_coi_results, pcr_results = run_daily_analytics()

Starting Daily Options Analytics

[Step 1] Loading data...
Loading: 281125_bhavcopy.csv
Loaded 32377 records
Loading previous: 271125_bhavcopy.csv

[Step 2] Computing Options Change in Open Interest...
Computed 615 option symbols

[Step 3] Computing Futures Change in Open Interest...
Computed 0 future symbols

[Step 4] Computing Put-Call Ratios...
Computed 615 symbol-expiry combinations

[Step 5] Exporting to Excel...


C:\Users\bkaly\AppData\Local\Temp\ipykernel_12944\1644114607.py:26: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  result['Previous_COI'].fillna(0, inplace=True)



Excel file saved successfully: C:\Users\bkaly\Documents\Trading\analytics_output\OptionsAnalytics_20251211_102808.xlsx

SUMMARY - Top 5 Options COI Changes:
    Symbol     Expiry  Current_COI  COI_Change  COI_Change_Pct
      GAIL 2025-12-30     45964800    44207100     2515.053763
    SUZLON 2025-12-30     13544000     5736000       73.463115
       LTF 2025-12-30      7978056     5202692      187.459807
GMRAIRPORT 2025-12-30      5580000     4868550      684.313725
SAMMAANCAP 2025-12-30      8853700     3710900       72.157191

SUMMARY - Top 5 Futures COI Changes:
Empty DataFrame
Columns: [Symbol, Expiry, Current_COI, COI_Change, COI_Change_Pct]
Index: []

SUMMARY - Top 5 Put-Call Ratios (PCR OI):
   Symbol     Expiry  CE_OI  PE_OI    PCR_OI
   360ONE 2026-01-27    500  11500 23.000000
SOLARINDS 2026-01-27     50   1150 23.000000
     INFY 2026-02-24   1200  24400 20.333333
    TECHM 2026-01-27  21600 135000  6.250000
 BOSCHLTD 2026-01-27    100    625  6.250000

Daily Analytics Com

### Optional: View Results in Notebook

In [7]:
# Display options COI results
print("Options COI Summary:")
print(options_coi_results.head(10))

Options COI Summary:
         Symbol      Expiry  Current_COI  Previous_COI  COI_Change  \
184        GAIL  2025-12-30     45964800       1757700    44207100   
533      SUZLON  2025-12-30     13544000       7808000     5736000   
343         LTF  2025-12-30      7978056       2775364     5202692   
190  GMRAIRPORT  2025-12-30      5580000        711450     4868550   
497  SAMMAANCAP  2025-12-30      8853700       5142800     3710900   
178  FEDERALBNK  2025-12-30      2610000      -1080000     3690000   
600        VEDL  2025-12-30      9315000       5787950     3527050   
298      JIOFIN  2025-12-30      5372100       2246600     3125500   
78          BEL  2025-12-30      4832175       1872450     2959725   
223        HFCL  2025-12-30      4760100       1838250     2921850   

     COI_Change_Pct  
184     2515.053763  
533       73.463115  
343      187.459807  
190      684.313725  
497       72.157191  
178     -341.666667  
600       60.937810  
298      139.121339  
78       1

In [8]:
# Display futures COI results
print("Futures COI Summary:")
print(futures_coi_results.head(10))

Futures COI Summary:
Empty DataFrame
Columns: [Symbol, Expiry, Current_COI, Previous_COI, COI_Change, COI_Change_Pct]
Index: []


In [9]:
# Display PCR results
print("Put-Call Ratio Summary:")
print(pcr_results.head(10))

Put-Call Ratio Summary:
       Symbol      Expiry     CE_OI  CE_Volume   CE_COI     PE_OI  PE_Volume  \
0      360ONE  2025-12-30    954000       4036   156000    473500        993   
1      360ONE  2026-01-27       500          0        0     11500          1   
2      360ONE  2026-02-24         0          0        0         0          0   
3         ABB  2025-12-30    670875       5282    57625    612625       2547   
4         ABB  2026-01-27      5375          0        0      9250         30   
5         ABB  2026-02-24         0          0        0         0          0   
6   ABCAPITAL  2025-12-30  15639500       7431  1853800  13999600       5460   
7   ABCAPITAL  2026-01-27    241800         38    80600    291400         72   
8   ABCAPITAL  2026-02-24         0          0        0         0          0   
9  ADANIENSOL  2025-12-30   2099925       6952   322650   1262250       1435   

    PE_COI     PCR_OI  PCR_Volume   PCR_COI  
0    18500   0.496331    0.246036  0.118590  
1  